In [22]:
#################################################################
## Model 3 : Continuous-time - ADP - Policy gradient ##
#################################################################

import numpy as np
import random

# Parameter
T = 7.0  # Total time periods
Nt = 200  # time steps
dt = T / Nt  # step size
ages = np.linspace(18, 25, Nt)

# grid of state variables
asset_min, asset_max = -20000 * dt, 50000 * dt
asset_grid = np.round(np.linspace(asset_min, asset_max, 51), 2)
skill_grid = np.round(np.linspace(0.0, 1.0, 201), 2)
educ_levels = np.round(np.linspace(0.0, 4.0, 201), 2)
exp_grid = np.round(np.linspace(0.0, 7.0, 201), 2)
parent_income_grid = [0, 1]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}

delta = 0.95
gamma = delta ** dt  # continuous discount per dt

# wage, tuition & shock
def wage(educ, skill):
    return (5000 + 5000 * educ + 3000 * skill) * dt  # wage per dt
sigma_eps = 1500 * np.sqrt(dt)  # shock in each step
tuition = 3000 * dt  # tuition in each step

# probability of education success
def education_success_prob(ability, educ):
    educ = np.clip(educ, 0.0, 4.0)  # make sure between 0 and 4
    base_success = {
        'low': 0.7,
        'medium': 0.8,
        'high': 0.9
    }[ability]

    # each +1.0 in educ will decrease the probability of 0.1
    success_rate = base_success - 0.1 * (educ / 1.0)
    success_rate = max(success_rate, 0.0)  # make sure won't be negative

    return success_rate

def crra(c, rho=0.95):
    return np.sign(c) * (np.abs(c) ** (1 - rho)) / (1 - rho)

def behavior_utility(action_type):
    return {'study': -0.05 * dt, 'work': -10 * dt, 'delay': -2 * dt}[action_type]

def terminal_reward(asset, educ, skill):
    return crra(max(asset, 1.0), rho=0.95) + 0.5 * educ

actions = ['study', 'work', 'delay']
ACTIONS = [(act, an) for act in actions for an in asset_grid]

# Feature Vector
def feature_vector(state, action):
    age, educ, asset, skill, exp, p_inc, ability = state
    act, asset_next = action
    ability_idx = ability_map[ability]
    return np.array([
        (age - 18) / T,
        educ / 4,
        (asset - asset_min) / (asset_max - asset_min),
        skill,
        exp / 7,
        p_inc,
        ability_idx / 2,
        (asset_next - asset_min) / (asset_max - asset_min),
        actions.index(act) / 2
    ])

# Softmax Policy
def softmax_policy(state, theta, actions=ACTIONS):
    logits = np.array([np.dot(theta, feature_vector(state, a)) for a in actions])
    exps = np.exp(logits - np.max(logits))
    return exps / np.sum(exps)

# Legal Actions
def legal_actions(state):
    age, educ, asset_now, skill, exp, p_inc, ability = state
    legal = []
    for action in ACTIONS:
        act, asset_next = action
        if asset_next < asset_min:
            continue
        base_income = wage(educ, skill) if act == 'work' else 0
        cost = tuition if act == 'study' else 0
        worst_shock = -0.09 * sigma_eps if act == 'work' else 0
        income = base_income + worst_shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            continue
        legal.append(action)
    return legal

# choose actions
def choose_action(state, theta):
    legal = legal_actions(state)
    if not legal:
        return None
    probs = softmax_policy(state, theta, actions=legal)
    return legal[np.random.choice(len(legal), p=probs)]

# Trajectory Simulation
def simulate_trajectory(s0, theta):
    trajectory = []
    state = s0
    for step in range(Nt):
        action = choose_action(state, theta)
        if action is None:
            break

        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action

        base_income = 0
        cost = 0
        if act == 'work':
            base_income = wage(educ, skill)
        elif act == 'study':
            cost = tuition

        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            break

        reward = crra(consumption) + behavior_utility(act)

        # state involvement
        skill_next = skill + (0.05 if act == 'study' else 0.02) * dt
        skill_next = min(1.0, max(0.0, skill_next))

        exp_next = exp + (dt if act == 'work' else 0)
        exp_next = min(6.0, exp_next)

        educ_next = educ
        if act == 'study':
          p_succ = education_success_prob(ability, educ)
          if random.random() < p_succ :
                educ_next = min(4.0, educ + 1 * dt)

        ability_next = random.choices(['low', 'medium', 'high'],weights=[0.1, 0.6, 0.3] if ability == 'medium' else ([0.5, 0.4, 0.1] if ability == 'low' else [0.0, 0.2, 0.8]))[0]

        next_state = (
            age + dt,
            closest_grid_value(educ_next, educ_levels),
            closest_grid_value(asset_next, asset_grid),
            closest_grid_value(skill_next, skill_grid),
            closest_grid_value(exp_next, exp_grid),
            p_inc,
            ability_next
        )

        trajectory.append((state, action, reward))
        state = next_state

        if state[0] >= 25.0:
            break

    final_rew = terminal_reward(state[2], state[1], state[3])
    trajectory.append((state, None, final_rew))
    return trajectory

def closest_grid_value(x, grid):
    return float(grid[np.argmin(np.abs(grid - x))])

# Returns
def compute_returns(traj, gamma=gamma):
    G = 0
    returns = []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

# Compute Policy Gradient
def compute_policy_gradient(traj, returns, theta):
    grads = np.zeros_like(theta)
    for (s, a, _), G in zip(traj, returns):
        if a is None:
            continue
        probs = softmax_policy(s, theta)
        phi = np.array([feature_vector(s, act) for act in ACTIONS])
        grad_logpi = feature_vector(s, a) - np.dot(probs, phi)
        grads += G * grad_logpi
    return grads

# Policy Gradient training
def train_policy_gradient(theta_init, alpha=0.01, epochs=200, episodes_per_epoch=50):
    theta = theta_init.copy()
    history = []
    for epoch in range(epochs):
        grads_all = []
        returns_all = []
        for _ in range(episodes_per_epoch):
            s0 = (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high')
            traj = simulate_trajectory(s0, theta)
            R = compute_returns(traj)
            grad = compute_policy_gradient(traj, R, theta)
            grads_all.append(grad)
            returns_all.append(R[0])
        theta += alpha * np.mean(grads_all, axis=0)
        avg_return = np.mean(returns_all)
        history.append(avg_return)
        if epoch % 10 == 0:
            print(f"[Epoch {epoch}] Average Return = {avg_return:.2f}")
    return theta, history

# Print Policy trajectory
def print_policy_trajectory(theta, start_state):
    print("\n🧭 Policy Trajectory from state", start_state)
    state = start_state
    for step in range(Nt):
        action = choose_action(state, theta)
        if action is None:
            print(f"Age {state[0]:.2f}: State = {state} → ❌ No Action Available")
            break

        current_ability = state[6]

        if current_ability == 'low':
            next_ability = 'medium'  # 0.5 low → 0.4 medium → 0.1 high
        elif current_ability == 'medium':
            next_ability = 'medium'  # 0.6 medium最大
        else:  # high
            next_ability = 'high'    # 0.8 high最大

        print(f"Age {state[0]:.2f}: {current_ability} → {next_ability}, State = {state}, Action = {action}")

        act, asset_next = action
        next_skill = min(1.0, state[3] + (0.05 if act == 'study' else 0.02) * dt)
        next_exp = min(6.0, state[4] + (dt if act == 'work' else 0))

        if act == 'study':
            p_succ = education_success_prob(state[6], state[1])
            if p_succ >= 0.5:
                next_educ = state[1] + 1 * dt
            else:
                next_educ = state[1]
        else:
            next_educ = state[1]

        next_state = (
            state[0] + dt,
            closest_grid_value(next_educ, educ_levels),
            closest_grid_value(asset_next, asset_grid),
            closest_grid_value(next_skill, skill_grid),
            closest_grid_value(next_exp, exp_grid),
            state[5],
            next_ability
        )

        state = next_state

        if state[0] >= 25.0:
            break


np.random.seed(42)
theta0 = np.random.randn(9)
theta_trained, return_history = train_policy_gradient(theta0)

# Estimate the expected utility
s0 = (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high')
vals = [compute_returns(simulate_trajectory(s0, theta_trained))[0] for _ in range(200)]
print(f"\nEstimated V(s0) ≈ {np.mean(vals):.2f}")

print_policy_trajectory(theta_trained, start_state=s0)


[Epoch 0] Average Return = 157.52
[Epoch 10] Average Return = 145.41
[Epoch 20] Average Return = 130.95
[Epoch 30] Average Return = 130.79
[Epoch 40] Average Return = 139.08
[Epoch 50] Average Return = 119.27
[Epoch 60] Average Return = 155.03
[Epoch 70] Average Return = 124.53
[Epoch 80] Average Return = 132.70
[Epoch 90] Average Return = 139.89
[Epoch 100] Average Return = 149.88
[Epoch 110] Average Return = 128.65
[Epoch 120] Average Return = 122.58
[Epoch 130] Average Return = 135.42
[Epoch 140] Average Return = 150.61
[Epoch 150] Average Return = 101.60
[Epoch 160] Average Return = 124.80
[Epoch 170] Average Return = 155.78
[Epoch 180] Average Return = 116.93
[Epoch 190] Average Return = 115.60

Estimated V(s0) ≈ 125.06

🧭 Policy Trajectory from state (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high')
Age 18.00: high → high, State = (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high'), Action = ('delay', np.float64(-700.0))
Age 18.04: high → high, State = (18.035, 0.0, -700.0, 0.0, 0.0, 1, 'high'), Action = 

In [1]:
############################################################
## Model 3 : Continuous-time - ADP - Linear VFA ##
############################################################

import numpy as np
import random

# Parameter
T = 7.0
Nt = 200
dt = T / Nt
ages = np.linspace(18, 25, Nt)

asset_min, asset_max = -20000 * dt, 50000 * dt
asset_grid = np.round(np.linspace(asset_min, asset_max, 51), 2)
skill_grid = np.round(np.linspace(0.0, 1.0, 201), 2)
educ_levels = np.round(np.linspace(0.0, 4.0, 201), 2)
exp_grid = np.round(np.linspace(0.0, 7.0, 201), 2)
parent_income_grid = [0, 1]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}

delta = 0.95
gamma = delta ** dt

ability_transition = {
    'low':    {'low': 0.5, 'medium': 0.4, 'high': 0.1},
    'medium': {'low': 0.1, 'medium': 0.6, 'high': 0.3},
    'high':   {'low': 0.0, 'medium': 0.2, 'high': 0.8}
}


def wage(educ, skill):
    return (5000 + 5000 * educ + 3000 * skill) * dt

sigma_eps = 1500 * np.sqrt(dt)
tuition = 3000 * dt

def education_success_prob(ability, educ):
    educ = np.clip(educ, 0.0, 4.0)
    base_success = {'low': 0.7, 'medium': 0.8, 'high': 0.9}[ability]
    success_rate = base_success - 0.1 * (educ / 1.0)
    return max(success_rate, 0.0)

def crra(c, rho=0.95):
    return np.sign(c) * (np.abs(c) ** (1 - rho)) / (1 - rho)

def behavior_utility(action_type):
    return {'study': -0.05 * dt, 'work': -10 * dt, 'delay': -2 * dt}[action_type]

def terminal_reward(asset, educ, skill):
    return crra(max(asset, 1.0), rho=0.95) + 0.5 * educ

def closest_grid_value(x, grid):
    return float(grid[np.argmin(np.abs(grid - x))])

# Feature & Value Function
def state_feature_vector(state):
    age, educ, asset, skill, exp, p_inc, ability = state
    ability_idx = ability_map[ability]
    return np.array([
        (age - 18) / T,
        educ / 4,
        (asset - asset_min) / (asset_max - asset_min),
        skill,
        exp / 7,
        p_inc,
        ability_idx / 2,
    ])

def value_function(state, theta):
    return np.dot(theta, state_feature_vector(state))

# Legal Actions & Action Simulation
actions = ['study', 'work', 'delay']
ACTIONS = [(act, an) for act in actions for an in asset_grid]

def legal_actions(state):
    age, educ, asset_now, skill, exp, p_inc, ability = state
    legal = []
    for action in ACTIONS:
        act, asset_next = action
        if asset_next < asset_min:
            continue
        base_income = wage(educ, skill) if act == 'work' else 0
        cost = tuition if act == 'study' else 0
        worst_shock = -0.09 * sigma_eps if act == 'work' else 0
        income = base_income + worst_shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            continue
        legal.append(action)
    return legal

def best_action(state, theta):
    legal = legal_actions(state)
    if not legal:
        return None
    best_a = None
    best_value = -np.inf
    for action in legal:
        act, asset_next = action
        # Simulate transition
        age, educ, asset_now, skill, exp, p_inc, ability = state
        base_income = wage(educ, skill) if act == 'work' else 0
        cost = tuition if act == 'study' else 0
        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        reward = crra(consumption) + behavior_utility(act)
        next_skill = skill + (0.05 if act == 'study' else 0.02) * dt
        next_skill = min(1.0, max(0.0, next_skill))
        next_exp = exp + (dt if act == 'work' else 0)
        next_exp = min(7.0, next_exp)
        next_educ = educ
        if act == 'study':
            p_succ = education_success_prob(ability, educ)
            educ_next = educ + 1 * dt if random.random() < p_succ else educ
        else:
            educ_next = educ
        next_ability = random.choices(['low', 'medium', 'high'],weights=[0.1, 0.6, 0.3] if ability == 'medium' else ([0.5, 0.4, 0.1] if ability == 'low' else [0.0, 0.2, 0.8]))[0]

        next_state = (
            age + dt,
            closest_grid_value(next_educ, educ_levels),
            closest_grid_value(asset_next, asset_grid),
            closest_grid_value(next_skill, skill_grid),
            closest_grid_value(next_exp, exp_grid),
            p_inc,
            next_ability
        )

        if next_state[0] >= 25.0:
            future_value = terminal_reward(next_state[2], next_state[1], next_state[3])
        else:
            future_value = value_function(next_state, theta)
        value = reward + gamma * future_value
        if value > best_value:
            best_value = value
            best_a = action
    return best_a

# ------------------ 6. Trajectory Simulation ------------------
def simulate_trajectory(s0, theta):
    trajectory = []
    state = s0
    for _ in range(Nt):
        action = best_action(state, theta)
        if action is None:
            break
        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action
        base_income = wage(educ, skill) if act == 'work' else 0
        cost = tuition if act == 'study' else 0
        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        reward = crra(consumption) + behavior_utility(act)
        next_skill = skill + (0.05 if act == 'study' else 0.02) * dt
        next_skill = min(1.0, max(0.0, next_skill))
        next_exp = exp + (dt if act == 'work' else 0)
        next_exp = min(7.0, next_exp)
        next_educ = educ
        if act == 'study':
            p_succ = education_success_prob(ability, educ)
            educ_next = educ + 1 * dt if random.random() < p_succ else educ
        else:
            educ_next = educ
        next_ability = random.choices(['low', 'medium', 'high'],weights=[0.1, 0.6, 0.3] if ability == 'medium' else ([0.5, 0.4, 0.1] if ability == 'low' else [0.0, 0.2, 0.8]))[0]
        next_state = (
            age + dt,
            closest_grid_value(next_educ, educ_levels),
            closest_grid_value(asset_next, asset_grid),
            closest_grid_value(next_skill, skill_grid),
            closest_grid_value(next_exp, exp_grid),
            p_inc,
            next_ability
        )
        trajectory.append((state, reward, next_state))
        state = next_state
        if state[0] >= 25.0:
            break
    return trajectory

# Training
def train_value_iteration(theta_init, alpha=0.01, epochs=200, episodes_per_epoch=50):
    theta = theta_init.copy()
    history = []
    for epoch in range(epochs):
        all_losses = []
        for _ in range(episodes_per_epoch):
            s0 = (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high')
            traj = simulate_trajectory(s0, theta)
            for (s, r, s_next) in traj:
                target = r
                if s_next[0] >= 25.0:
                    target += 0
                else:
                    target += gamma * value_function(s_next, theta)
                prediction = value_function(s, theta)
                loss = 0.5 * (target - prediction) ** 2
                grad = (target - prediction) * state_feature_vector(s)
                theta += alpha * grad
                all_losses.append(loss)
        avg_loss = np.mean(all_losses)
        history.append(avg_loss)
        if epoch % 10 == 0:
            print(f"[Epoch {epoch}] Average Loss = {avg_loss:.6f}")
    return theta, history

# Print greedy policy trajectory
def print_greedy_trajectory(theta, start_state):
    print("\n🧭 Policy Trajectory from state", start_state)
    state = start_state
    for _ in range(Nt):
        action = best_action(state, theta)
        if action is None:
            print(f"Age {state[0]:.2f}: {state} → ❌ No legal action")
            break

        print(f"Age {state[0]:.2f}: State = {state}, Action = {action}")

        # state involvement
        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action

        next_skill = min(1.0, max(0.0, skill + (0.05 if act == 'study' else 0.02) * dt))
        next_exp = min(7.0, exp + (dt if act == 'work' else 0))

        if act == 'study':
            p_succ = education_success_prob(ability, educ)
            next_educ = educ + 1 * dt if p_succ >= 0.5 else educ
        else:
            next_educ = educ

        next_ability = max(
            ability_transition[ability],
            key=ability_transition[ability].get
        )

        next_state = (
            age + dt,
            closest_grid_value(next_educ, educ_levels),
            closest_grid_value(asset_next, asset_grid),
            closest_grid_value(next_skill, skill_grid),
            closest_grid_value(next_exp, exp_grid),
            p_inc,
            next_ability
        )

        state = next_state

        if state[0] >= 25.0:
            break


np.random.seed(42)
theta0 = np.random.randn(7)
theta_trained, loss_history = train_value_iteration(theta0)

# Estimate expected utility
s0 = (18.0, 0.0, 0.0, 0.0, 0.0, 1, 'high')
V0_star = value_function(s0, theta_trained)
print(f"\n⭐ Estimated V_0* ≈ {V0_star:.2f}")
print_greedy_trajectory(theta_trained, s0)


[Epoch 0] Average Loss = 3487.402959
[Epoch 10] Average Loss = 263.993021
[Epoch 20] Average Loss = 237.200281
[Epoch 30] Average Loss = 238.366341
[Epoch 40] Average Loss = 254.471349
[Epoch 50] Average Loss = 250.355247


KeyboardInterrupt: 